In [2]:
import dspy
import os
from dotenv import load_dotenv
from openai import OpenAI

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
client = OpenAI(api_key=API_KEY)

In [5]:
lm = dspy.LM("openai/o3", api_key=API_KEY, temperature=1, max_tokens=20_000)
dspy.configure(lm=lm)

In [6]:
class DataPromptSignature(dspy.Signature):
    """Signature for generating GPT-4 data-generation prompt."""
    data_description = dspy.InputField(desc="Description of the labeled, in-distribution data you want GPT-4 to generate")
    data_generation_prompt = dspy.OutputField(desc="Detailed prompt text to give GPT-4 for generating data")

class DataPromptGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_prompt = dspy.Predict(signature=DataPromptSignature)

In [ ]:
# class DataPromptGenerator(dspy.Module):
#     def __init__(self):
#         super().__init__()
#         self.generate_prompt = dspy.Predict(
#             inputs=["data_description"],
#             outputs=["data_generation_prompt"],
#             instructions=(
#                 "Given a description of some type of labeled data, generate a detailed prompt "
#                 "that can be used with GPT-4 to create realistic in-distribution synthetic data. "
#                 "The prompt should be clear, specify the format, and provide examples if possible."
#             )
#         )

In [7]:
class DataPromptGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_prompt = dspy.Predict(signature=DataPromptSignature)

In [8]:
prompt_generator = DataPromptGenerator()

desc = "A dataset of 10 realistic video game chat contents from a player in the DOTA 2 game, the content should be formed by combining all the messages from one player in a match and seperating the messages with a period. There will be some summary statistics given such as the mean and standard deviation of each datapoint as well as the most frequent words for each label given to assist with the generations. The labels should be for toxic and non-toxic players, with 1 representing non-toxic and -1 representing toxic players. The output should only be the text and label, no extra information"

res = prompt_generator.generate_prompt(data_description=desc)
print(res.data_generation_prompt)

You are a data-generation engine that must output 10 labeled examples of DOTA 2 in-game chat logs, each representing all messages sent by ONE player during ONE match.  

Output requirements
1. Produce exactly 10 lines.  
2. Each line = the player’s combined chat messages, followed by a tab character “\t”, followed by the label.  
   • Label 1  → non-toxic player  
   • Label -1 → toxic player  
3. No line-breaks inside a datapoint except the final newline that separates one datapoint from the next.  
4. No extra commentary, numbering, or formatting other than specified.

Content rules
• Combine the player’s individual messages in chronological order, separating each original message with a period and a space.  
• Length per datapoint: 35–60 words (≈ 45 ± 10).  
• Mix in typical DOTA 2 terms: “mid”, “bot”, “top”, “ward”, “push”, “gank”, “rosh”, “bkb”, hero names, item names, etc.  
• Use informal gamer style, occasional abbreviations (“gg”, “wp”, “gj”, “omw”), light profanity when appro

In [9]:
print(res.data_generation_prompt)

You are a data-generation engine that must output 10 labeled examples of DOTA 2 in-game chat logs, each representing all messages sent by ONE player during ONE match.  

Output requirements
1. Produce exactly 10 lines.  
2. Each line = the player’s combined chat messages, followed by a tab character “\t”, followed by the label.  
   • Label 1  → non-toxic player  
   • Label -1 → toxic player  
3. No line-breaks inside a datapoint except the final newline that separates one datapoint from the next.  
4. No extra commentary, numbering, or formatting other than specified.

Content rules
• Combine the player’s individual messages in chronological order, separating each original message with a period and a space.  
• Length per datapoint: 35–60 words (≈ 45 ± 10).  
• Mix in typical DOTA 2 terms: “mid”, “bot”, “top”, “ward”, “push”, “gank”, “rosh”, “bkb”, hero names, item names, etc.  
• Use informal gamer style, occasional abbreviations (“gg”, “wp”, “gj”, “omw”), light profanity when appro

In [10]:
res.data_generation_prompt

'You are a data generator tasked with creating a small, realistic, in-distribution dataset that mimics post-match chat logs from individual DOTA 2 players.  \n\nGENERAL REQUIREMENTS  \n1. Produce exactly 10 separate datapoints.  \n2. Output the result as a JSON array.  \n3. Each element in the array must be an object with two keys:  \n   • "text": a single string containing ALL messages that one player wrote during a match, in chronological order, each message separated by a period followed by a space (“. ”).  \n   • "label": an integer where 1 means non-toxic and ‑1 means toxic.  \n4. Deliver roughly balanced classes (about 5 toxic, 5 non-toxic).  \n5. Length constraint per “text”: 40 – 120 words after concatenation (about 300 – 900 characters).  \n6. Style & vocabulary must be recognisably DOTA 2 in-game chat: hero names (e.g., “Pudge”, “Invoker”), map call-outs (“bot rune”, “mid tower”), shorthand (“gg”, “gl hf”, “juked”), and gaming slang (“gank”, “farm”, “ward”).  \n7. Toxic examp

In [18]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": res.data_generation_prompt}
    ]
)

In [12]:
print(response.choices[0].message.content)

Here's a dataset with 10 examples of DOTA 2 chat content, labeled as non-toxic (1) or toxic (-1):

1. Non-Toxic (1):
   - "Nice last hit, Juggernaut! Keep it up and we'll outfarm them!"

2. Non-Toxic (1):
   - "Can we group up for a smoke gank? I think we can catch them off guard."

3. Non-Toxic (1):
   - "Great warding, thanks for the vision support. Makes it much easier to roam."

4. Non-Toxic (1):
   - "No worries about the early deaths, let's focus on farming and come back stronger in mid-game!"

5. Non-Toxic (1):
   - "Good job everyone! That was a tough fight but we played it well. Let's keep the momentum."

6. Toxic (-1):
   - "Are you blind, or just stupid? How did you miss that stun?"

7. Toxic (-1):
   - "GG, this team is hopeless. I'm just gonna AFK in base now."

8. Toxic (-1):
   - "Wow, our mid is feeding like it's a buffet. Report this noob."

9. Toxic (-1):
   - "Can you stop pretending like you know how to play? You're ruining the game."

10. Toxic (-1):
    - "Nice, a

In [19]:
print(response.choices[0].message.content)

"chat","label"  
"gg wp, team! Nice work on those last fights!",1  
"Why are you still in mid? You're useless 😂",1  
"pls remember to ward jungle, we're getting ganked too much",1  
"wtf are you doing?? uninstall pls",–1  
"awesome rotations, support! keep it up!",1  
"ez game, you guys were trash 🤣",–1  
"good hustle, we'll get them next time!",1  
"stop feeding, you noob!",–1  
"that's the worst build I've ever seen 🤦",–1  
"thanks for the carry, team! such a fun match!",1  
